# Rzut ukośny – analiza numeryczna

Notebook przygotowany na podstawie pliku `generuj_wykresy.py`. Uruchom komórki kolejno od góry. Wykresy zostaną zapisane w katalogu `files/rzut_ukosny/`.

In [4]:
%pip -q install numpy matplotlib scipy

In [5]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Dane zgodne z uwagami w Issues: dwie pierwsze i dwie ostatnie cyfry indeksu.
INDEX_FIRST_TWO = 33
INDEX_LAST_TWO = 24

v0 = float(INDEX_FIRST_TWO)          # m/s
alpha_deg = float(INDEX_LAST_TWO)    # stopnie
alpha_rad = np.radians(alpha_deg)
g = 9.81                             # m/s^2
m = 1.0                              # kg
b = 0.15                             # kg/s = N s/m

OUT = Path("files/rzut_ukosny")
OUT.mkdir(parents=True, exist_ok=True)

BLUE, ORANGE, GREEN, PURPLE = "#2463A6", "#D47722", "#168578", "#8455A4"
INK = "#233044"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelcolor": INK, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.2,
    "lines.linewidth": 2.3, "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

def save(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

# Ruch idealny.
t_flight = 2 * v0 * np.sin(alpha_rad) / g
t = np.linspace(0, t_flight, 500)
vx0, vy0 = v0 * np.cos(alpha_rad), v0 * np.sin(alpha_rad)
x = vx0 * t
y = vy0 * t - 0.5 * g * t**2
vx = np.full_like(t, vx0)
vy = vy0 - g * t
speed = np.hypot(vx, vy)

# Model z liniowym oporem: F_op = -b v.
gamma = b / m
def rhs(_t, s):
    px, py, pvx, pvy = s
    return [pvx, pvy, -gamma * pvx, -g - gamma * pvy]

def hit_ground(_t, s):
    return s[1]
hit_ground.terminal = True
hit_ground.direction = -1

sol = solve_ivp(
    rhs, (0, 100), [0, 0, vx0, vy0], events=hit_ground,
    rtol=1e-10, atol=1e-11, max_step=0.01, dense_output=False
)
tz = sol.t
xz, yz, vxz, vyz = sol.y
speedz = np.hypot(vxz, vyz)

# Schemat fizyczny.
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.set(xlim=(-0.5, 4), ylim=(-0.5, 3))
ax.axis("off")
ax.annotate("", xy=(3.5, 0), xytext=(-0.2, 0), arrowprops=dict(arrowstyle="->", lw=1.5, color=INK))
ax.annotate("", xy=(0, 2.7), xytext=(0, -0.2), arrowprops=dict(arrowstyle="->", lw=1.5, color=INK))
ax.text(3.6, -0.1, "x (poziom)")
ax.text(-0.3, 2.7, "y (pion)")
ax.plot(0, 0, "o", color=BLUE, ms=8)
ax.annotate("", xy=(1.8*np.cos(alpha_rad), 1.8*np.sin(alpha_rad)), xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", lw=2.5, color=ORANGE))
ax.text(1.9*np.cos(alpha_rad), 1.9*np.sin(alpha_rad), r"$v_0$", color=ORANGE, fontsize=14)
arc = np.linspace(0, alpha_rad, 50)
ax.plot(0.6*np.cos(arc), 0.6*np.sin(arc), color=GREEN, lw=1.5)
ax.text(0.7*np.cos(alpha_rad/2), 0.7*np.sin(alpha_rad/2), r"$\alpha$", color=GREEN, fontsize=13)
ax.set_title(f"Schemat rzutu ukośnego: v₀ = {v0:.0f} m/s, α = {alpha_deg:.0f}°", loc="left")
save(fig, "schemat")

# Trajektoria idealna – osobny wykres.
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
ax.plot(x, y, color=BLUE, label="Model idealny")
ax.set(xlabel="Odległość x [m]", ylabel="Wysokość y [m]",
       title="Trajektoria rzutu ukośnego bez oporu")
ax.legend()
save(fig, "trajektoria_idealna")

# Porównanie trajektorii.
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
ax.plot(x, y, color=BLUE, label="Bez oporu")
ax.plot(xz, yz, color=ORANGE, ls="--", label=f"Z oporem (b = {b:.2f} kg/s)")
ax.set(xlabel="Odległość x [m]", ylabel="Wysokość y [m]", title="Porównanie trajektorii")
ax.legend()
save(fig, "trajektoria_porownanie")

# Składowe prędkości i szybkość.
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True, layout="constrained")
axes[0].plot(t, vx, color=BLUE, label="$v_x$ – bez oporu")
axes[0].plot(tz, vxz, color=ORANGE, ls="--", label="$v_x$ – z oporem")
axes[0].set_ylabel("$v_x$ [m/s]")
axes[0].legend()
axes[1].plot(t, vy, color=BLUE, label="$v_y$ – bez oporu")
axes[1].plot(tz, vyz, color=ORANGE, ls="--", label="$v_y$ – z oporem")
axes[1].set_ylabel("$v_y$ [m/s]")
axes[1].legend()
axes[2].plot(t, speed, color=BLUE, label="Szybkość – bez oporu")
axes[2].plot(tz, speedz, color=ORANGE, ls="--", label="Szybkość – z oporem")
axes[2].set(xlabel="Czas t [s]", ylabel="Szybkość [m/s]")
axes[2].legend()
axes[0].set_title("Składowe prędkości i szybkość w czasie", loc="left")
save(fig, "predkosc_czas")

# Bilans energii.
Ek = 0.5 * m * (vx**2 + vy**2)
Ep = m * g * y
Et = Ek + Ep
Ekz = 0.5 * m * (vxz**2 + vyz**2)
Epz = m * g * yz
Etz = Ekz + Epz

fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
ax.plot(t, Et, color=BLUE, label="Całkowita – bez oporu")
ax.plot(tz, Etz, color=ORANGE, ls="--", label="Całkowita – z oporem")
ax.plot(tz, Ekz, color=GREEN, ls=":", label="Kinetyczna – z oporem")
ax.plot(tz, Epz, color=PURPLE, ls=":", label="Potencjalna – z oporem")
ax.set(xlabel="Czas t [s]", ylabel="Energia [J]", title="Bilans energii mechanicznej")
ax.legend()
save(fig, "energia_czas")

# Wyraźny wykres dyssypacji energii.
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
ax.plot(tz, Etz[0] - Etz, color=ORANGE, label="Ubytek energii mechanicznej")
ax.set(xlabel="Czas t [s]", ylabel="Ubytek energii [J]",
       title="Dyssypacja energii wskutek oporu powietrza")
ax.legend()
save(fig, "strata_energii")

# Przestrzeń fazowa.
fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
ax.plot(y, vy, color=BLUE, label="Bez oporu")
ax.plot(yz, vyz, color=ORANGE, ls="--", label="Z oporem")
ax.set(xlabel="Wysokość y [m]", ylabel="Prędkość pionowa $v_y$ [m/s]",
       title="Portret fazowy w osi pionowej")
ax.legend()
save(fig, "przestrzen_fazowa")

# Wpływ współczynnika oporu.
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
ax.plot(x, y, color=BLUE, label="b = 0 kg/s")
for b_value, color in zip([0.05, 0.15, 0.30], [GREEN, ORANGE, PURPLE]):
    gamma_value = b_value / m
    def rhs_b(_t, s):
        return [s[2], s[3], -gamma_value*s[2], -g-gamma_value*s[3]]
    sol_b = solve_ivp(rhs_b, (0, 100), [0, 0, vx0, vy0],
                      events=hit_ground, rtol=1e-9, atol=1e-10, max_step=0.01)
    ax.plot(sol_b.y[0], sol_b.y[1], color=color, ls="--",
            label=f"b = {b_value:.2f} kg/s")
ax.set(xlabel="Odległość x [m]", ylabel="Wysokość y [m]",
       title="Wpływ współczynnika oporu na trajektorię")
ax.legend()
save(fig, "wplyw_b")

print(f"Parametry: v0={v0:.1f} m/s, alpha={alpha_deg:.1f} deg, g={g:.2f} m/s^2, m={m:.1f} kg, b={b:.2f} kg/s")
print(f"Model idealny: czas={t_flight:.3f} s, wysokość={np.max(y):.3f} m, zasięg={x[-1]:.3f} m")
print(f"Model z oporem: czas={tz[-1]:.3f} s, wysokość={np.max(yz):.3f} m, zasięg={xz[-1]:.3f} m")
print(f"Wykresy zapisano w: {OUT.resolve()}")


Parametry: v0=33.0 m/s, alpha=24.0 deg, g=9.81 m/s^2, m=1.0 kg, b=0.15 kg/s
Model idealny: czas=2.736 s, wysokość=9.182 m, zasięg=82.496 m
Model z oporem: czas=2.572 s, wysokość=8.092 m, zasięg=64.323 m
Wykresy zapisano w: /content/files/rzut_ukosny
